# 📊 SÍNTESE VISUAL 1 — Benchmark Completo de pAUC@0.1

In [ ]:
import json, sys, numpy as np
sys.path.insert(0, '..')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

METRICS_DIR = Path('experiments_results/metrics')
FIGURES_DIR = Path('experiments_results/figures')

plt.rcParams.update({
    'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'text.color':'#f0f6fc','axes.labelcolor':'#f0f6fc',
    'xtick.color':'#8b949e','ytick.color':'#8b949e',
    'axes.edgecolor':'#30363d','grid.color':'#30363d','grid.alpha':0.5,
})

def load(nb_id):
    p = METRICS_DIR/f'{nb_id}_results.json'
    return json.load(open(p)) if p.exists() else {}

def best_pauc(data):
    for key in ['avg_prot_a','pauc01']:
        if key in data: return float(data[key])
    if 'protocol_a' in data: return float(data['protocol_a'].get('pauc01',0))
    if 'spec' in data: return float(data['spec'].get('avg_prot_a',0))
    if 'models' in data:
        vals = [v.get('pauc01',0) for v in data['models'].values() if isinstance(v,dict)]
        if vals: return float(max(vals))
    if 'model' in data:
        m = data['model']
        if isinstance(m,dict): return float(m.get('pauc01',0))
    if 'ranking' in data:
        vals = [v.get('pauc01',0) for v in data['ranking'].values()]
        if vals: return float(max(vals))
    return 0.0

all_data = {f'nb{i:02d}': load(f'nb{i:02d}') for i in range(1,29)}
print(f"✅ {sum(1 for v in all_data.values() if v)} resultados carregados")


In [ ]:
# Coleta pAUC de todos os notebooks
nbs = [f'nb{i:02d}' for i in range(1,29)]
pauc_vals = [best_pauc(all_data[nb]) for nb in nbs]
labels = [nb.upper() for nb in nbs]

fig, ax = plt.subplots(figsize=(16, 6))
colors = ['#2ea043' if p >= 0.80 else '#da3633' for p in pauc_vals]
bars = ax.bar(labels, pauc_vals, color=colors, alpha=0.85, edgecolor='none')
ax.axhline(0.80, color='#f0883e', ls='--', lw=2, label='Limite mínimo 0.80')
for bar, val in zip(bars, pauc_vals):
    if val > 0:
        ax.text(bar.get_x()+bar.get_width()/2, val+0.01, f'{val:.2f}',
                ha='center', va='bottom', fontsize=7, color='white')
ax.set_ylim(0, 1.1); ax.set_ylabel('pAUC@0.1')
ax.set_title('Benchmark Completo: pAUC@0.1 por Notebook (NB01–NB28)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, axis='y', alpha=0.4)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('experiments_results/figures/visual1_pauc_all.png', dpi=130,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"Notebooks acima do limite 0.80: {sum(1 for p in pauc_vals if p>=0.80)}/28")
